# US Data

References : [Chapter 5 : PCE (BEA)](https://www.bea.gov/resources/methodologies/nipa-handbook/pdf/chapter-05.pdf)

Careful to NIPA levels ; the same tables 2.4.3, 2.4.4 and 2.4.5 without the "U" that stands for "Underlying Details" are different aggregation levels. Those one are the most frequently used, while Lansing and Shapiro use the "Underlying Details" tables (129 components).

## Definition
Personal Consumption Expenditures (PCE) is a measure of the spending on goods and services by people of the United States, constructed and reported by the Bureau of Economic Analysis (BEA). According to the BEA, PCE accounts for about two-thirds of domestic spending and is a significant driver of gross domestic product (GDP).

Its main aggregates are...

We collect :
- **Table 2.4.4U :** Price Indexes for Personal Consumption Expenditures by Type of Product
- **Table 2.4.5U :** Personal Consumption Expenditures by Type of Product

chained value (2017=100) of PCE components indices, real and nominal, as well as quantities at a monthly frequency. Data is seasonally adjusted. 

In [38]:
# Librairies
import json
import pandas as pd
from fismi import sdxData as sdx
import numpy as np

### Settings

In [39]:
# Pick the desired level ("4" is the disagregation level used in Shapiro et Al., 2026)
niv = "4"

### Index sub-components


In [40]:
with open('/Users/lea_gosselin/.openbb_platform/user_settings.json', 'r') as file:
    keys = json.load(file)

In [41]:
url = "https://apps.bea.gov/api/data"
userid = keys["credentials"].get("bea_api_key")

# Load PCE data from BEA

# Query data
price_raw   = sdx.getBeaData("U20404", userid)   # Table 2.4.4U : Price Indexes for Personal Consumption Expenditures by Type of Product
weights_raw = sdx.getBeaData("U20405", userid)   # Table 2.4.5U : Personal Consumption Expenditures by Type of Product


In [42]:
# Load PCE components levels (not available online)
levelPce = pd.read_excel("PCEComponentsLevel.xlsx", dtype={"LineNumber": str, "Level": str})
levelPce[levelPce["Level"]==niv]

,LineNumber,Level,LineDescription,SeriesCode
5,6,4,New autos,DNEARA
8,9,4,New light trucks,DNWTRA
12,13,4,Used autos,DNPURA
16,17,4,Used light trucks,DUTRRA
20,21,4,Tires,DTATRA
...,...,...,...,...
345,346,4,"Nonprofit hospitals, gross output",DHSORA
346,347,4,"Nonprofit nursing homes, gross output",DNXORA
357,358,4,Outpatient services to households,DOUSRA
358,359,4,Nonprofit hospitals services to househ...,DNPHRA


In [43]:
# How many components ? 2.4.4
price_raw.loc[price_raw["LineNumber"].isin(levelPce.loc[levelPce["Level"]==niv, "LineNumber"]), ["LineNumber","LineDescription","SeriesCode"]].drop_duplicates()

,LineNumber,LineDescription,SeriesCode
4055,6,New autos,DNEARG
6488,9,New light trucks,DNWTRG
8700,13,Used autos,DNPURG
11944,17,Used light trucks,DUTRRG
14612,21,Tires,DTATRG
...,...,...,...
263231,346,"Nonprofit hospitals, gross output",DHSORG
264042,347,"Nonprofit nursing homes, gross output",DNXORG
272963,358,Outpatient services to households,DOUSRG
273774,359,Nonprofit hospitals services to households,DNPHRG


In [44]:
# How many components ? 2.4.5
weights_raw.loc[weights_raw["LineNumber"].isin(levelPce.loc[levelPce["Level"]==niv, "LineNumber"]), ["LineNumber","LineDescription","SeriesCode"]].drop_duplicates()

,LineNumber,LineDescription,SeriesCode
4055,6,New autos,DNEARC
6488,9,New light trucks,DNWTRC
8700,13,Used autos,DNPURC
11944,17,Used light trucks,DUTRRC
14612,21,Tires,DTATRC
...,...,...,...
264853,346,"Nonprofit hospitals, gross output",DHSORC
265664,347,"Nonprofit nursing homes, gross output",DNXORC
274585,358,Outpatient services to households,DOUSRC
275396,359,Nonprofit hospitals services to households,DNPHRC


In [45]:
# Merge data on LineNumber - Index
price_raw.loc[price_raw["LineNumber"].isin(levelPce.loc[levelPce["Level"]==niv, "LineNumber"]), ["LineNumber","LineDescription","SeriesCode"]].drop_duplicates()
price_raw = price_raw.merge(
    levelPce[["LineNumber","Level"]],
    how="left",
    on="LineNumber")
price_raw

,TIME_PERIOD,LineNumber,LineDescription,SeriesCode,DataValue,Level
0,1959-01-01,1,Personal consumption expenditures,DPCERG,15.164,NaN
1,1959-02-01,1,Personal consumption expenditures,DPCERG,15.179,NaN
2,1959-03-01,1,Personal consumption expenditures,DPCERG,15.189,NaN
3,1959-04-01,1,Personal consumption expenditures,DPCERG,15.219,NaN
4,1959-05-01,1,Personal consumption expenditures,DPCERG,15.227,NaN
...,...,...,...,...,...,...
303405,2026-03-01,402,Market-based PCE excluding food and energy,DPCXRG,126.356,NaN
303406,2026-04-01,402,Market-based PCE excluding food and energy,DPCXRG,126.729,NaN
303407,2026-05-01,402,Market-based PCE excluding food and energy,DPCXRG,127.018,NaN
303408,2026-06-01,402,Market-based PCE excluding food and energy,DPCXRG,127.247,NaN


In [46]:
# Merge data on LineNumber - Quantites
weights_raw.loc[weights_raw["LineNumber"].isin(levelPce.loc[levelPce["Level"]==niv, "LineNumber"]), ["LineNumber","LineDescription","SeriesCode"]].drop_duplicates()
weights_raw = weights_raw.merge(
    levelPce[["LineNumber","Level"]],
    how="left",
    on="LineNumber")
weights_raw

,TIME_PERIOD,LineNumber,LineDescription,SeriesCode,DataValue,Level
0,1959-01-01,1,Personal consumption expenditures,DPCERC,306091,NaN
1,1959-02-01,1,Personal consumption expenditures,DPCERC,309554,NaN
2,1959-03-01,1,Personal consumption expenditures,DPCERC,312702,NaN
3,1959-04-01,1,Personal consumption expenditures,DPCERC,312193,NaN
4,1959-05-01,1,Personal consumption expenditures,DPCERC,316130,NaN
...,...,...,...,...,...,...
305027,2026-03-01,402,Market-based PCE excluding food and energy,DPCXRC,16334610,NaN
305028,2026-04-01,402,Market-based PCE excluding food and energy,DPCXRC,16422689,NaN
305029,2026-05-01,402,Market-based PCE excluding food and energy,DPCXRC,16563570,NaN
305030,2026-06-01,402,Market-based PCE excluding food and energy,DPCXRC,16673260,NaN


In [47]:
# Weights PCE (share of the expenditure in the total, as current $) 
df_price   = price_raw[price_raw["Level"]==niv].drop_duplicates(subset=["TIME_PERIOD","LineNumber"]).pivot(index="TIME_PERIOD", columns="LineDescription", values="DataValue")
df_weights = weights_raw[weights_raw["Level"]==niv].drop_duplicates(subset=["TIME_PERIOD","LineNumber"]).pivot(index="TIME_PERIOD", columns="LineDescription", values="DataValue")

### Weights

The share of expenditures in the total index is expressed in current Dollar, such as : 
$$
w_{i,t} = \frac{PCE_{i,t}}{PCE_{Total,t}}
$$

In [48]:
# Compute weights over time
df_weights = df_weights.div(df_weights.sum(axis=1, skipna=True),  axis=0)

In [49]:
# PCE data
PCE = price_raw[
    (price_raw["SeriesCode"]=="DPCERG")
    ].drop_duplicates(subset=["TIME_PERIOD","LineNumber"]).pivot(index="TIME_PERIOD", columns="LineDescription", values="DataValue")
PCE

LineDescription,Personal consumption expenditures
TIME_PERIOD,
1959-01-01,15.164
1959-02-01,15.179
1959-03-01,15.189
1959-04-01,15.219
1959-05-01,15.227
...,...
2026-03-01,130.403
2026-04-01,130.946
2026-05-01,131.576


In [36]:
# Add DPCERG : PCE total

# Append HICP Total
df_price = df_price.merge(
    PCE,
    how="left",
    left_index=True,
    right_index=True
)
df_price.to_parquet(f"input/USItemsLvl{niv}.parquet")

# Weights...
df_weights.to_parquet(f"input/USnormedWeightsLvl{niv}.parquet")